## Settings

In [9]:
## auto reload modules
%reload_ext autoreload
%autoreload 2

## Dependencies

In [10]:
## libraries
import sys
import logging
from pathlib import Path

## path
root = Path.cwd().resolve().parent
sys.path.insert(0, str(root))

## modules
from src.estimators.factories import load_estimators
from src.data.builders import (
    load_processed_data,
    load_perturbed_data
)
from src.evaluators.perturbing import (
    eval_perturbed_frontier,
    eval_perturbed_alignment,
    eval_perturbed_consensus,
    find_perturbed_max,
    spec_marginal_delta,
    stat_perturbed_tost,
)
from src.evaluators.config import (
    FEAT_X,
    FEAT_Z,
    TARGET
)

## Initialization

In [13]:
## reproducibility
N_REPEATS = 30
RANDOM_STATE = 42

## load data and models
_disable = logging.root.manager.disable
logging.disable(logging.INFO)
try:
    data_proc = load_processed_data()
    data_pert_all = load_perturbed_data()
    data_pert = {
        key: data_pert_all[key]
        for key in [
            "network_perturbed", 
            "invariants_perturbed", 
            "process_perturbed", 
            "signatures_perturbed"
        ]
        if key in data_pert_all
    }
finally:
    logging.disable(_disable)
    models = load_estimators(random_state = RANDOM_STATE)

## view data shape
n_obs, n_feat = data_proc.shape
print(f"Original Data: {n_feat} features, {n_obs} observations")
print("Perturbed Data:")
for json_key, methods in data_pert.items():
    print(f"  {json_key}:")
    for method, intense in methods.items():
        pert_df = next(iter(intense.values()))
        n_r, n_c = pert_df.shape
        print(f"   - {method}: {n_c} features, {n_r} observations")

Original Data: 32 features, 25 observations
Perturbed Data:
  network_perturbed:
   - node_sample: 24 features, 24 observations
   - rewire: 24 features, 24 observations
   - sparsify: 24 features, 24 observations
  invariants_perturbed:
   - jitter: 24 features, 24 observations
   - noise: 24 features, 24 observations
   - subset: 24 features, 24 observations
  process_perturbed:
   - bootstrapping: 10 features, 24 observations
   - scaling: 10 features, 24 observations
   - smoothing: 10 features, 24 observations
  signatures_perturbed:
   - jitter: 10 features, 24 observations
   - noise: 10 features, 24 observations
   - subset: 10 features, 24 observations


## Frontier Evaluation
Four input perturbation types (network, invariants, process, signatures) that each consist of three targeted techniques are applied while the target remains fixed. Model robustness is evaluated at maximum perturbation intensity using LOGO-CV (domain), where models are trained using original data and predict on the perturbations.

In [14]:
## perturbation evaluation
if "results_perturbed" not in globals():
    results_perturbed, _ = eval_perturbed_frontier(
        data = data_proc,
        models = models,
        data_pert = data_pert,
        feat_x = FEAT_X,
        feat_z = FEAT_Z,
        target = TARGET,
    )

Perturbation training: 100%|██████████| 756/756 [39:07<00:00,  3.10s/job]  


## Frontier Equivalence Test
Each perturbation method is tested for equivalence against the unperturbed baseline EI using two one-sided tests (TOST) with paired Wilcoxon signed-rank statistics.

- **$H_0$**: The absolute difference meets or exceeds the margin ($|\Delta| \ge \delta$).
- **$H_1$**: The absolute difference falls strictly within the margin ($|\Delta| < \delta$).

Rejecting $H_0$ establishes that the perturbation effect is negligibly small relative to $\delta$.

### Maximum-Intensity Protocol
Only the highest intensity setting per perturbation method enters the test. Baseline and perturbed EI values are paired on (model $\times$ domain), matching the falsification pipeline's pairing convention. This yields one paired comparison per model $\times$ domain $\times$ method and prevents a dense intensity sweep from inflating sample size.

### Empirical Test Specification
The equivalence margin $\delta$ is derived from baseline variability across learners on the original EI results. Half the interquartile range of the baseline EI values sets $\delta$, anchoring the margin to natural performance dispersion rather than to perturbation effects.

In [ ]:
## maximum intensity per perturbation method
results_perturbed_max = find_perturbed_max(
    results = results_perturbed
)

## empirical delta specification
delta = spec_marginal_delta(
    results = results_perturbed,
    feat_value = ["ei"],
    track = "frozen",
    method = "iqr",
    scale = 1.0,  ## iqr range
    decimals = 3
)

## tost equivalence test across all perturbation methods
results = stat_perturbed_tost(
    results = results_perturbed_max,
    feat_value = ["ei"],
    feat_group = ["perturbation", "method"],
    track = "frozen",
    delta = delta,
    decimals = 3
)

display(results)

All four perturbation methods achieve TOST equivalence at maximum intensity. Effect sizes are negligible across all methods. The largest median shift (0.006, process) consumes less than one-quarter of the equivalence margin. EI is robust to aggressive corruption of network structure and its graph invariant vector representation, as well as stochastic process and its process signature vector representation.

## Target-Alignment Evaluation
Whereas the EI test above probes whether perturbations distort the predictive frontier, the target-alignment test probes whether perturbations distort the *ranking* of predictions against the observed target. Predictions are generated under the same frozen LOGO-CV (domain) protocol used for EI, then scored by the composite consensus index CI = f(rho, rbo, dcr) per (model x domain).


In [ ]:
## perturbation alignment evaluation
if "results_perturbed_alignment" not in globals():
    results_perturbed_alignment = eval_perturbed_alignment(
        data = data_proc,
        models = models,
        data_pert = data_pert,
        feat_x = FEAT_X,
        feat_z = FEAT_Z,
        target = TARGET,
        n_repeats = N_REPEATS,
        random_state = RANDOM_STATE,
    )


## Target-Alignment Equivalence Test
Each perturbation method is tested for equivalence against the unperturbed baseline CI using two one-sided tests (TOST) with paired Wilcoxon signed-rank statistics.

- **$H_0$**: The absolute difference meets or exceeds the margin ($|\Delta| \ge \delta$).
- **$H_1$**: The absolute difference falls strictly within the margin ($|\Delta| < \delta$).

Rejecting $H_0$ establishes that the perturbation effect is negligibly small relative to $\delta$.

### Maximum-Intensity Protocol
Only the highest intensity setting per perturbation method enters the test. Baseline and perturbed CI values are paired on (model $\times$ domain), matching the frontier EI test. This yields one paired comparison per model $\times$ domain $\times$ method and prevents a dense intensity sweep from inflating sample size.

### Empirical Test Specification
The equivalence margin $\delta$ is derived from baseline variability across learners on the original target-alignment CI results. Half the interquartile range of the baseline CI values sets $\delta$, anchoring the margin to natural performance dispersion rather than to perturbation effects.

In [ ]:
## maximum intensity per perturbation method
results_perturbed_alignment_max = find_perturbed_max(
    results = results_perturbed_alignment
)

## empirical delta specification on ci
delta_alignment = spec_marginal_delta(
    results = results_perturbed_alignment,
    feat_value = ["ci"],
    track = "frozen",
    method = "iqr",
    scale = 1.0,
    decimals = 3
)

## tost equivalence test across all perturbation methods
results_alignment = stat_perturbed_tost(
    results = results_perturbed_alignment_max,
    feat_value = ["ci"],
    feat_group = ["perturbation", "method"],
    feat_pairs = ["model", "group"],
    track = "frozen",
    delta = delta_alignment,
    decimals = 3
)

display(results_alignment)


## Pairwise Consensus Evaluation
The CI test above probes alignment between each model and the observed target; the pairwise consensus test probes alignment *between models*. Two models can each remain individually aligned with the target while drifting apart from one another, so this is a strictly stronger test of perturbation neutrality. Models are fit once on clean data via full-data repeated fits and then scored on each perturbed dataframe (frozen protocol, consistent with the EI and target-alignment tests). Pairwise consensus is computed across all $\binom{M}{2}$ model pairs on the resulting prediction vectors.


In [ ]:
## perturbation pairwise consensus evaluation
if "results_perturbed_consensus" not in globals():
    results_perturbed_consensus = eval_perturbed_consensus(
        data = data_proc,
        models = models,
        data_pert = data_pert,
        feat_x = FEAT_X,
        feat_z = FEAT_Z,
        target = TARGET,
        n_repeats = N_REPEATS,
        random_state = RANDOM_STATE,
    )


## Pairwise Consensus Equivalence Test
Each perturbation method is tested for equivalence against the unperturbed baseline pairwise CI using two one-sided tests (TOST) with paired Wilcoxon signed-rank statistics.

- **$H_0$**: The absolute difference meets or exceeds the margin ($|\Delta| \ge \delta$).
- **$H_1$**: The absolute difference falls strictly within the margin ($|\Delta| < \delta$).

Rejecting $H_0$ establishes that the perturbation effect is negligibly small relative to $\delta$.

### Maximum-Intensity Protocol
Only the highest intensity setting per perturbation method enters the test. Baseline and perturbed pairwise CI values are paired on (model$_i$ $\times$ model$_j$), with one comparison retained for each model pair $\times$ method. This keeps the pairing rule parallel to the frontier and target-alignment tests while preventing a dense intensity sweep from inflating sample size.

### Empirical Test Specification
The equivalence margin $\delta$ is derived from baseline variability across model pairs on the original pairwise-consensus CI results. Half the interquartile range of the baseline pairwise CI values sets $\delta$, anchoring the margin to natural performance dispersion rather than to perturbation effects.

In [ ]:
## maximum intensity per perturbation method
results_perturbed_consensus_max = find_perturbed_max(
    results = results_perturbed_consensus
)

## empirical delta specification on pairwise ci
delta_consensus = spec_marginal_delta(
    results = results_perturbed_consensus,
    feat_value = ["ci"],
    track = "frozen",
    method = "iqr",
    scale = 1.0,
    decimals = 3
)

## tost equivalence test across all perturbation methods
results_consensus = stat_perturbed_tost(
    results = results_perturbed_consensus_max,
    feat_value = ["ci"],
    feat_group = ["perturbation", "method"],
    feat_pairs = ["model_i", "model_j", "group"],
    track = "frozen",
    delta = delta_consensus,
    decimals = 3
)

display(results_consensus)
